# Chakraborty–Stokes adaptive nighttime-lights modelling and reliability-qualified disaster anomalies for Samar–Leyte

This notebook tests whether the adaptive nighttime-lights forecasting workflow of Chakraborty and Stokes (2023) detects the shock and recovery associated with Super Typhoon Haiyan, and whether its output changes when gap-filled continuity is replaced by reliability-qualified observations.

The analysis follows four linked questions:

1. **Replication:** What Haiyan anomaly is produced from gap-filled Black Marble using the published 30-day smoothing and multi-step neural-network forecasting design?
2. **Reliability qualification:** Can the same model be fitted to the supplied GHSL-masked, directly observed `DNB_BRDF_Corrected_NTL` summaries without filling cloud-driven gaps?
3. **Observability:** Which model outputs are supported by sufficient temporal and spatial observation coverage?
4. **Comparison:** How do anomaly direction, severity, detection delay, and recovery timing vary between gap-filled and reliability-qualified inputs?

The gap-filled series is a methodological benchmark and diagnostic comparison. The reliability-qualified branch is the evidential branch. NTL anomalies indicate departures in electricity-dependent nocturnal activity; they do not directly measure electricity restoration or community recovery.

Primary reference: [Chakraborty and Stokes (2023)](https://doi.org/10.1016/j.rse.2023.113818), *Remote Sensing of Environment*, 298, 113818. An [open manuscript](https://arxiv.org/abs/2306.08501) is also available.


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import io
import re
import warnings
import zipfile

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)


In [2]:
# ============================================================
# 2. PATHS, EVENT WINDOWS, AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"

GAP_FILLED_CSV = DATA_DIR / "Region VIII_NTL_VNP46A2.csv"
GHSL_DIR = DATA_DIR / "ghsl_masked_ntl_samar_leyte"
GHSL_ZIP = DATA_DIR / "ghsl_masked_ntl_samar_leyte.zip"

# Haiyan and retrospective training design
EVENT_DATE = pd.Timestamp("2013-11-08")
TRAINING_END = EVENT_DATE - pd.Timedelta(days=90)
DISPLAY_START = EVENT_DATE - pd.Timedelta(days=365)
DISPLAY_END = EVENT_DATE + pd.Timedelta(days=540)
HAIYAN_WINDOW_END = EVENT_DATE + pd.Timedelta(days=180)

# Chakraborty–Stokes settings
ROLLING_DAYS = 30
INPUT_WINDOW = 60
OUTPUT_WINDOW = 30
TRAIN_FRACTION = 0.80
BATCH_SIZE = 64
MODEL_EPOCHS = {
    "FCNN": 70,
    "CNN": 90,
    "LSTM": 25,
}
ENSEMBLE_WEIGHTS = {
    "FCNN": 0.30,
    "CNN": 0.20,
    "LSTM": 0.50,
}
ANOMALY_TOP_PERCENT = 25
RANDOM_SEED = 42

# Reliability and recovery settings
MIN_OBSERVED_DAYS_30D = 18
VALID_PIXEL_THRESHOLD_PCT = 50.0
RECOVERY_PERSISTENCE_DAYS = 14

# Signals
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
RQ_SIGNAL = "NTL_mean"

# Plot colours retained from Notebook 2
OBSERVED_COLOR = "#0091FF"
PREDICTED_COLOR = "#000000"
GAP_FILLED_COLOR = "#FF0000"
RQ_COLOR = "#00C54F"
EVENT_LINE_COLOR = "#0057FF"
ANOMALY_COLOR = "#C62828"
LOW_SUPPORT_COLOR = "#A8B6CC"

SC_COLORSCALE = [
    [0.00, "#F7FCF5"],
    [0.10, "#E5F5E0"],
    [0.25, "#C7E9C0"],
    [0.50, "#74C476"],
    [0.75, "#238B45"],
    [1.00, "#005A32"],
]

if not GAP_FILLED_CSV.exists():
    raise FileNotFoundError(
        f"Gap-filled CSV not found: {GAP_FILLED_CSV}"
    )

if not GHSL_DIR.exists() and not GHSL_ZIP.exists():
    raise FileNotFoundError(
        "Provide either the extracted GHSL directory or its ZIP archive: "
        f"{GHSL_DIR} or {GHSL_ZIP}"
    )

print(f"Gap-filled input: {GAP_FILLED_CSV}")
print(
    "Reliability-qualified input: "
    f"{GHSL_DIR if GHSL_DIR.exists() else GHSL_ZIP}"
)
print(f"Training ends: {TRAINING_END.date()}")


Gap-filled input: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/Region VIII_NTL_VNP46A2.csv
Reliability-qualified input: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/ghsl_masked_ntl_samar_leyte
Training ends: 2013-08-10


## 1. Published method and Samar–Leyte adaptation

Chakraborty and Stokes derive a daily, area-weighted urban NTL series from gap-filled VNP46A2, smooth it with a 30-day rolling average, and train three neural networks on a stable pre-change phase. A 60-day input sequence predicts the next 30 days. The FCNN, 1-D CNN, and LSTM models use Adam optimization, mean absolute error loss, a batch size of 64, and 70, 90, and 25 epochs, respectively. Each date can receive multiple forecasts from overlapping output windows; the published workflow takes their median. The three predictions are then combined using fixed ensemble weights.

Change is represented by the residual

$$
r_t = x_t - \hat{x}_{t,ens},
$$

and a time step is anomalous when $r_t^2 > \tau$, where $\tau$ selects the largest 25% of squared errors. Severity is $|r_t|$, direction is $\mathrm{sgn}(r_t)$, and the error trajectory is used to examine post-change recovery.

The article assigns 0.5 to LSTM and 0.3 to the fully connected model. Its remaining 0.2 label repeats “LSTM”; this notebook interprets that typographical duplication as CNN because the surrounding text contrasts stable LSTM forecasts with less stable CNN forecasts. This gives FCNN/CNN/LSTM weights of 0.3/0.2/0.5.

Two transparent adaptations are required here:

- VIIRS begins in January 2012, leaving less than two years before Haiyan rather than the paper’s preferred minimum of three years. Training therefore ends 90 days before landfall and uses all eligible earlier sequences.
- The GHSL files are reliability-qualified daily summary tables, not dense gap-filled pixel series. Their 30-day mean uses only directly observed days and is retained only when at least 18 of 30 days are observed. No missing daily NTL value is interpolated. Spatial completeness remains a separate quality flag.

The supplied GHSL tables contain summary statistics rather than pixel-level radiances. Any percentile clipping applied upstream is preserved, but a new pixel-level 95th-percentile clamp cannot be reconstructed from these tables. `NTL_p95` is retained as a diagnostic and is not substituted for mean NTL.


## 2. Data preparation

The two input branches are prepared separately. The regional benchmark uses `Gap_Filled_DNB_BRDF_Corrected_NTL`. The GHSL branch uses `NTL_mean`, which summarizes directly observed `DNB_BRDF_Corrected_NTL` within each thematic mask. Their spatial domains are related but not identical: the benchmark covers Region VIII, while the reliability-qualified summaries describe the supplied Samar–Leyte GHSL supports.


In [3]:
# ============================================================
# 3.1 LOAD GAP-FILLED REGION VIII BLACK MARBLE
# ============================================================

gap_filled_daily = pd.read_csv(
    GAP_FILLED_CSV,
    parse_dates=["date"],
)

required_gap_columns = {
    "date",
    DNB_BAND,
    GAP_FILLED_BAND,
}

missing_gap_columns = required_gap_columns.difference(
    gap_filled_daily.columns
)

if missing_gap_columns:
    raise KeyError(
        f"Missing gap-filled columns: {sorted(missing_gap_columns)}"
    )

gap_filled_daily = (
    gap_filled_daily
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .set_index("date")
    .asfreq("D")
)

gap_filled_daily["ntl_raw"] = pd.to_numeric(
    gap_filled_daily[GAP_FILLED_BAND],
    errors="coerce",
)

gap_filled_daily["direct_dnb_raw"] = pd.to_numeric(
    gap_filled_daily[DNB_BAND],
    errors="coerce",
)

gap_filled_daily["observed_day"] = (
    gap_filled_daily["ntl_raw"].notna()
)

gap_filled_daily["temporal_support_pct"] = (
    gap_filled_daily["observed_day"]
    .rolling(
        ROLLING_DAYS,
        min_periods=1,
    )
    .mean()
    .mul(100)
)

# The paper applies a 30-day rolling mean to its gap-filled series.
gap_filled_daily["ntl_30d"] = (
    gap_filled_daily["ntl_raw"]
    .rolling(
        ROLLING_DAYS,
        min_periods=ROLLING_DAYS,
    )
    .mean()
)

gap_filled_daily["spatial_completeness_pct"] = np.nan
gap_filled_daily["input_type"] = "Gap-filled Black Marble"

display(
    gap_filled_daily[
        [
            "ntl_raw",
            "direct_dnb_raw",
            "ntl_30d",
            "temporal_support_pct",
        ]
    ]
    .describe()
    .round(3)
)


,ntl_raw,direct_dnb_raw,ntl_30d,temporal_support_pct
count,4112.000,3503.000,3330.000,4162.000
mean,0.248,0.512,0.248,98.810
std,0.196,0.823,0.073,4.486
min,0.071,0.000,0.137,46.667
25%,0.180,0.212,0.198,100.000
50%,0.225,0.329,0.241,100.000
75%,0.274,0.505,0.282,100.000
max,10.086,18.058,0.716,100.000


In [4]:
# ============================================================
# 3.2 LOAD RELIABILITY-QUALIFIED GHSL-MASKED TABLES
# ============================================================

GHSL_MASKS = {
    "10-30": "G1 (codes 10–30)",
    "11-30": "G2 (codes 11–30)",
    "12-30": "G3 (codes 12–30)",
    "13-30": "G4 (codes 13–30)",
    "21-30": "G5 (codes 21–30)",
    "22-30": "G6 (codes 22–30)",
    "23-30": "G7 (codes 23–30)",
    "30": "G8 (code 30)",
}


def ghsl_code_from_name(file_name):
    match = re.search(
        r"DNBBRDF_(.+?)_stats\.csv$",
        Path(file_name).name,
    )

    if match is None:
        raise ValueError(
            f"Cannot identify the GHSL mask from {file_name}"
        )

    return match.group(1)


def load_ghsl_tables():
    tables = {}

    if GHSL_DIR.exists():
        sources = sorted(GHSL_DIR.glob("*.csv"))

        for source in sources:
            code_value = ghsl_code_from_name(source.name)
            tables[code_value] = pd.read_csv(
                source,
                parse_dates=["date"],
            )

    else:
        with zipfile.ZipFile(GHSL_ZIP) as archive:
            sources = sorted(
                name
                for name in archive.namelist()
                if name.endswith(".csv")
                and not name.startswith("__MACOSX/")
            )

            for source in sources:
                code_value = ghsl_code_from_name(source)

                with archive.open(source) as stream:
                    tables[code_value] = pd.read_csv(
                        io.TextIOWrapper(
                            stream,
                            encoding="utf-8-sig",
                        ),
                        parse_dates=["date"],
                    )

    return tables


ghsl_raw = load_ghsl_tables()

missing_masks = set(GHSL_MASKS).difference(ghsl_raw)

if missing_masks:
    raise KeyError(
        f"Missing GHSL mask tables: {sorted(missing_masks)}"
    )

print(f"Loaded {len(ghsl_raw)} GHSL thematic-mask tables.")


Loaded 8 GHSL thematic-mask tables.


In [5]:
# ============================================================
# 3.3 PREPARE RELIABILITY-QUALIFIED 30-DAY SERIES
# ============================================================

rq_daily = {}
rq_audit_rows = []

for code_value, mask_label in GHSL_MASKS.items():
    frame = (
        ghsl_raw[code_value]
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .set_index("date")
        .asfreq("D")
    )

    required_columns = {
        RQ_SIGNAL,
        "NTL_p95",
        "Valid_px",
    }

    missing_columns = required_columns.difference(frame.columns)

    if missing_columns:
        raise KeyError(
            f"{mask_label} is missing {sorted(missing_columns)}"
        )

    for column in required_columns:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    maximum_valid_pixels = frame["Valid_px"].max()

    frame["ntl_raw"] = frame[RQ_SIGNAL].where(
        frame["Valid_px"].gt(0)
    )

    frame["observed_day"] = frame["ntl_raw"].notna()

    frame["spatial_completeness_pct"] = (
        frame["Valid_px"]
        .div(maximum_valid_pixels)
        .mul(100)
    )

    frame["temporal_support_pct"] = (
        frame["observed_day"]
        .rolling(
            ROLLING_DAYS,
            min_periods=1,
        )
        .mean()
        .mul(100)
    )

    frame["ntl_30d"] = (
        frame["ntl_raw"]
        .rolling(
            ROLLING_DAYS,
            min_periods=MIN_OBSERVED_DAYS_30D,
        )
        .mean()
    )

    frame["input_type"] = "Reliability-qualified DNB-BRDF"
    frame["ghsl_mask"] = mask_label
    rq_daily[mask_label] = frame

    haiyan_frame = frame.loc[
        EVENT_DATE:HAIYAN_WINDOW_END
    ]

    rq_audit_rows.append(
        {
            "GHSL mask": mask_label,
            "Start": frame.index.min().date(),
            "End": frame.index.max().date(),
            "Maximum valid pixels": int(maximum_valid_pixels),
            "Observed days (%)": 100 * frame["observed_day"].mean(),
            "Haiyan observed days (%)": (
                100 * haiyan_frame["observed_day"].mean()
            ),
            "Haiyan median spatial completeness (%)": (
                haiyan_frame["spatial_completeness_pct"].median()
            ),
            "Haiyan days at or above valid-pixel threshold (%)": (
                100
                * haiyan_frame["spatial_completeness_pct"]
                .ge(VALID_PIXEL_THRESHOLD_PCT)
                .mean()
            ),
        }
    )

rq_audit = pd.DataFrame(rq_audit_rows)

display(
    rq_audit.style.format(
        {
            "Observed days (%)": "{:.1f}",
            "Haiyan observed days (%)": "{:.1f}",
            "Haiyan median spatial completeness (%)": "{:.1f}",
            "Haiyan days at or above valid-pixel threshold (%)": "{:.1f}",
        }
    )
)


,GHSL mask,Start,End,Maximum valid pixels,Observed days (%),Haiyan observed days (%),Haiyan median spatial completeness (%),Haiyan days at or above valid-pixel threshold (%)
0,G1 (codes 10–30),2012-01-19,2025-07-21,88991,80.3,88.4,36.5,45.9
1,G2 (codes 11–30),2012-01-19,2025-07-21,87035,80.3,88.4,36.6,45.9
2,G3 (codes 12–30),2012-01-19,2025-07-21,30549,79.9,87.3,37.2,44.2
3,G4 (codes 13–30),2012-01-19,2025-07-21,15528,79.3,87.3,37.5,43.1
4,G5 (codes 21–30),2012-01-19,2025-07-21,11480,78.0,86.2,37.3,43.1
5,G6 (codes 22–30),2012-01-19,2025-07-21,2782,76.2,85.6,33.6,40.9
6,G7 (codes 23–30),2012-01-19,2025-07-21,1492,72.8,85.6,31.9,39.2
7,G8 (code 30),2012-01-19,2025-07-21,449,62.8,75.1,32.1,38.1


### 2.1 Input distinction and observability rule

The gap-filled branch supplies a reconstructed value on nearly every date. Its 30-day availability is therefore not evidence that the ground was observed. The reliability-qualified branch uses only daily GHSL-masked DNB-BRDF summaries with at least one valid pixel. A rolling value requires 18 observed days within the trailing 30-day window; the remaining dates stay missing.

`Valid_px` is converted to relative spatial completeness using the maximum valid-pixel count within the same fixed thematic mask. The 50% threshold is not used to erase additional days before modelling because doing so leaves no contiguous 60-day input plus 30-day output sequences before Haiyan. It is instead attached to every Haiyan metric as an interpretability flag. This preserves the distinction between model feasibility and evidential strength.


In [6]:
# ============================================================
# 4. INPUT TIME-SERIES OVERVIEW
# ============================================================

g7_label = GHSL_MASKS["23-30"]
g7_daily = rq_daily[g7_label]

figure_inputs = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.22, 0.78],
    subplot_titles=(
        "30-day temporal support",
        "Thirty-day nighttime-lights inputs",
    ),
)

support_dates = gap_filled_daily.index.union(g7_daily.index)

figure_inputs.add_trace(
    go.Heatmap(
        x=support_dates,
        y=[
            "Gap-filled availability",
            "G7 observed-day support",
        ],
        z=np.vstack(
            [
                gap_filled_daily["temporal_support_pct"]
                .reindex(support_dates)
                .to_numpy(),
                g7_daily["temporal_support_pct"]
                .reindex(support_dates)
                .to_numpy(),
            ]
        ),
        coloraxis="coloraxis",
        zsmooth=False,
        hoverongaps=False,
        hovertemplate=(
            "%{x|%d %b %Y}<br>"
            "%{y}: %{z:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

figure_inputs.add_trace(
    go.Scatter(
        x=gap_filled_daily.index,
        y=gap_filled_daily["ntl_30d"],
        mode="lines",
        name="Gap-filled Region VIII",
        line=dict(
            color=GAP_FILLED_COLOR,
            width=2.2,
        ),
        connectgaps=False,
    ),
    row=2,
    col=1,
)

figure_inputs.add_trace(
    go.Scatter(
        x=g7_daily.index,
        y=g7_daily["ntl_30d"],
        mode="lines",
        name=f"Reliability-qualified {g7_label}",
        line=dict(
            color=RQ_COLOR,
            width=2.2,
        ),
        connectgaps=False,
    ),
    row=2,
    col=1,
)

for row_number in (1, 2):
    figure_inputs.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color=EVENT_LINE_COLOR,
            width=2,
            dash="dash",
        ),
        row=row_number,
        col=1,
    )

figure_inputs.add_annotation(
    x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
    y=0.05,
    xref="x2",
    yref="y2 domain",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        color=EVENT_LINE_COLOR,
        size=16,
    ),
)

figure_inputs.update_xaxes(
    range=[DISPLAY_START, DISPLAY_END],
    title_text="Date",
    row=2,
    col=1,
)

figure_inputs.update_yaxes(
    title_text="Mean radiance<br>(nW cm⁻² sr⁻¹)",
    row=2,
    col=1,
)

figure_inputs.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=650,
    title=dict(
        text="Gap-filled continuity and reliability-qualified observability",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    coloraxis=dict(
        colorscale=SC_COLORSCALE,
        cmin=0,
        cmax=100,
        colorbar=dict(
            title="Support (%)",
            thickness=18,
        ),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=105, r=100, t=130, b=70),
    hovermode="x unified",
)

figure_inputs.show()


**Interpretation.** The 30-day mean suppresses short daily fluctuations in both branches. In the gap-filled branch, this produces a dense reconstructed trajectory. In the reliability-qualified branch, line breaks and weaker support identify observation-limited periods. A missing reliability-qualified value is not a low-light observation.


## 3. Adaptive multi-step forecasting

The following functions preserve the paper’s direct sequence-to-sequence design. Training pairs are eligible only when all 60 input and 30 output values are finite and the output ends on or before the retrospective training cutoff. The final 20% of eligible baseline sequences is retained for validation in chronological order.

Forecasts are generated for every finite 60-day input sequence, including sequences after the training phase. Each day receives up to 30 overlapping forecasts from each model, and their median is retained. No post-Haiyan target value is used to retrain a model.


In [7]:
# ============================================================
# 5.1 BUILD TRAINING AND FORECAST WINDOWS
# ============================================================

def build_training_windows(series):
    values = series.to_numpy(dtype=float)
    dates = series.index

    x_values = []
    y_values = []

    final_start = len(series) - INPUT_WINDOW - OUTPUT_WINDOW + 1

    for start in range(max(0, final_start)):
        input_end = start + INPUT_WINDOW
        output_end = input_end + OUTPUT_WINDOW

        x_window = values[start:input_end]
        y_window = values[input_end:output_end]

        if dates[output_end - 1] > TRAINING_END:
            continue

        if not np.isfinite(x_window).all():
            continue

        if not np.isfinite(y_window).all():
            continue

        x_values.append(x_window)
        y_values.append(y_window)

    if not x_values:
        raise ValueError(
            "No complete pre-Haiyan 60-day input and 30-day "
            "output windows are available."
        )

    return (
        np.asarray(x_values, dtype=np.float32),
        np.asarray(y_values, dtype=np.float32),
    )


def build_forecast_windows(series):
    values = series.to_numpy(dtype=float)
    dates = series.index

    x_values = []
    output_dates = []

    final_start = len(series) - INPUT_WINDOW - OUTPUT_WINDOW + 1

    for start in range(max(0, final_start)):
        input_end = start + INPUT_WINDOW
        output_end = input_end + OUTPUT_WINDOW
        x_window = values[start:input_end]

        if not np.isfinite(x_window).all():
            continue

        x_values.append(x_window)
        output_dates.append(
            dates[input_end:output_end]
        )

    if not x_values:
        raise ValueError(
            "No complete 60-day forecast inputs are available."
        )

    return (
        np.asarray(x_values, dtype=np.float32),
        output_dates,
    )


def aggregate_overlapping_forecasts(
    predicted_windows,
    output_dates,
):
    prediction_rows = []

    for dates, values in zip(output_dates, predicted_windows):
        prediction_rows.extend(
            zip(dates, values)
        )

    return (
        pd.DataFrame(
            prediction_rows,
            columns=["date", "prediction"],
        )
        .groupby("date")["prediction"]
        .median()
        .sort_index()
    )


In [10]:
! pip install tensorflow
! pip install keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 MB 14.0 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 18.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.8/566.8 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 17.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]9 [tensorflow]a]


In [11]:
# ============================================================
# 5.2 CHAKRABORTY–STOKES NEURAL-NETWORK ARCHITECTURES
# ============================================================

try:
    import tensorflow as tf

    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import (
        BatchNormalization,
        Conv1D,
        Dense,
        Dropout,
        Flatten,
        Input,
        LSTM,
        MaxPooling1D,
    )
    from tensorflow.keras.optimizers import Adam

except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This replication requires TensorFlow/Keras. Install TensorFlow "
        "in the notebook kernel before running the modelling sections."
    ) from exc


np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


def compile_model(model):
    model.compile(
        optimizer=Adam(),
        loss="mae",
    )
    return model


def build_fcnn():
    return compile_model(
        Sequential(
            [
                Input(shape=(INPUT_WINDOW,)),
                Dense(60, activation="relu"),
                Dropout(0.10),
                Dense(45, activation="relu"),
                Dropout(0.10),
                Dense(25, activation="relu"),
                Dropout(0.10),
                Dense(OUTPUT_WINDOW, activation="relu"),
                Dropout(0.10),
            ],
            name="FCNN",
        )
    )


def build_cnn():
    layers = [Input(shape=(INPUT_WINDOW, 1))]

    for filters, kernel_size in [
        (90, 9),
        (45, 9),
        (30, 6),
        (20, 6),
    ]:
        layers.extend(
            [
                Conv1D(
                    filters=filters,
                    kernel_size=kernel_size,
                    padding="same",
                    activation="relu",
                ),
                MaxPooling1D(
                    pool_size=2,
                    padding="same",
                ),
                BatchNormalization(),
                Dropout(0.10),
            ]
        )

    layers.extend(
        [
            Flatten(),
            Dense(20, activation="relu"),
            Dense(15, activation="relu"),
            Dense(OUTPUT_WINDOW, activation="relu"),
        ]
    )

    return compile_model(
        Sequential(
            layers,
            name="CNN",
        )
    )


def build_lstm():
    return compile_model(
        Sequential(
            [
                Input(shape=(INPUT_WINDOW, 1)),
                LSTM(
                    45,
                    activation="relu",
                    return_sequences=True,
                ),
                Dropout(0.10),
                LSTM(
                    30,
                    activation="relu",
                ),
                Dropout(0.10),
                Dense(30, activation="relu"),
                Dense(15, activation="relu"),
                Dense(OUTPUT_WINDOW, activation="relu"),
            ],
            name="LSTM",
        )
    )


Matplotlib is building the font cache; this may take a moment.


In [12]:
# ============================================================
# 5.3 TRAIN, FORECAST, AND FORM THE WEIGHTED ENSEMBLE
# ============================================================

def model_input(values, model_name):
    if model_name == "FCNN":
        return values

    return values[..., np.newaxis]


def fit_adaptive_ensemble(
    prepared_frame,
    series_label,
):
    series = prepared_frame["ntl_30d"].copy()
    x_all, y_all = build_training_windows(series)

    split_index = int(len(x_all) * TRAIN_FRACTION)

    if split_index == 0 or split_index == len(x_all):
        raise ValueError(
            f"Insufficient training/validation split for {series_label}."
        )

    x_train = x_all[:split_index]
    y_train = y_all[:split_index]
    x_validation = x_all[split_index:]
    y_validation = y_all[split_index:]

    models = {
        "FCNN": build_fcnn(),
        "CNN": build_cnn(),
        "LSTM": build_lstm(),
    }

    history_rows = []

    for model_name, model in models.items():
        print(
            f"{series_label}: training {model_name} "
            f"for {MODEL_EPOCHS[model_name]} epochs"
        )

        history = model.fit(
            model_input(x_train, model_name),
            y_train,
            validation_data=(
                model_input(x_validation, model_name),
                y_validation,
            ),
            epochs=MODEL_EPOCHS[model_name],
            batch_size=BATCH_SIZE,
            shuffle=False,
            verbose=0,
        )

        history_rows.append(
            {
                "Series": series_label,
                "Model": model_name,
                "Training windows": len(x_train),
                "Validation windows": len(x_validation),
                "Final training MAE": history.history["loss"][-1],
                "Final validation MAE": history.history["val_loss"][-1],
            }
        )

    x_forecast, output_dates = build_forecast_windows(series)

    result = prepared_frame[
        [
            "ntl_30d",
            "observed_day",
            "temporal_support_pct",
            "spatial_completeness_pct",
        ]
    ].copy()

    result = result.rename(
        columns={"ntl_30d": "observed"}
    )

    for model_name, model in models.items():
        predicted_windows = model.predict(
            model_input(x_forecast, model_name),
            verbose=0,
        )

        result[f"predicted_{model_name}"] = (
            aggregate_overlapping_forecasts(
                predicted_windows,
                output_dates,
            )
        )

    result["predicted_ensemble"] = sum(
        ENSEMBLE_WEIGHTS[model_name]
        * result[f"predicted_{model_name}"]
        for model_name in ENSEMBLE_WEIGHTS
    )

    result["residual"] = (
        result["observed"]
        - result["predicted_ensemble"]
    )

    result["squared_error"] = result["residual"].pow(2)

    ensemble_threshold = result["squared_error"].quantile(
        1 - ANOMALY_TOP_PERCENT / 100
    )

    result["ensemble_anomaly"] = (
        result["squared_error"]
        .gt(ensemble_threshold)
        .where(result["squared_error"].notna())
    )

    model_flags = []

    for model_name in models:
        model_residual = (
            result["observed"]
            - result[f"predicted_{model_name}"]
        )

        model_squared_error = model_residual.pow(2)
        model_threshold = model_squared_error.quantile(
            1 - ANOMALY_TOP_PERCENT / 100
        )

        model_flag = model_squared_error.gt(model_threshold).where(
            model_squared_error.notna()
        )

        result[f"anomaly_{model_name}"] = model_flag
        model_flags.append(model_flag.astype(float))

    result["decision_confidence_pct"] = (
        pd.concat(model_flags, axis=1)
        .mean(axis=1, skipna=False)
        .mul(100)
    )

    result["series_label"] = series_label

    training_report = pd.DataFrame(history_rows)

    return models, result, training_report


In [13]:
# ============================================================
# 5.4 HAIYAN CHANGE AND RECOVERY METRICS
# ============================================================

def first_persistent_recovery(
    recovery_pct,
    threshold_pct,
):
    reached = recovery_pct.ge(threshold_pct).where(
        recovery_pct.notna()
    )

    persistent = (
        reached.astype(float)
        .rolling(
            RECOVERY_PERSISTENCE_DAYS,
            min_periods=RECOVERY_PERSISTENCE_DAYS,
        )
        .sum()
        .eq(RECOVERY_PERSISTENCE_DAYS)
    )

    if not persistent.any():
        return pd.NaT

    persistent_end = persistent[persistent].index[0]

    return (
        persistent_end
        - pd.Timedelta(days=RECOVERY_PERSISTENCE_DAYS - 1)
    )


def summarize_haiyan(
    result,
    series_label,
    input_type,
):
    event_frame = result.loc[
        EVENT_DATE:HAIYAN_WINDOW_END
    ].copy()

    observed_event = event_frame.dropna(
        subset=[
            "observed",
            "predicted_ensemble",
            "residual",
        ]
    )

    if observed_event.empty:
        return {
            "Series": series_label,
            "Input": input_type,
            "Quality flag": "Not observable",
        }

    impact_date = observed_event["residual"].idxmin()
    impact_row = observed_event.loc[impact_date]
    impact_severity = max(
        -impact_row["residual"],
        0,
    )

    negative_anomalies = observed_event[
        observed_event["ensemble_anomaly"].eq(True)
        & observed_event["residual"].lt(0)
    ]

    first_detection = (
        negative_anomalies.index.min()
        if not negative_anomalies.empty
        else pd.NaT
    )

    if impact_severity > 0:
        post_impact = observed_event.loc[impact_date:].copy()

        post_impact["recovery_pct"] = (
            1
            - post_impact["residual"]
            .mul(-1)
            .clip(lower=0)
            .div(impact_severity)
        ).mul(100)

        t50_date = first_persistent_recovery(
            post_impact["recovery_pct"],
            50,
        )

        t80_date = first_persistent_recovery(
            post_impact["recovery_pct"],
            80,
        )

    else:
        t50_date = pd.NaT
        t80_date = pd.NaT

    temporal_observation_pct = (
        100 * event_frame["observed_day"].mean()
    )

    median_spatial_completeness = event_frame[
        "spatial_completeness_pct"
    ].median()

    days_at_valid_pixel_threshold = (
        100
        * event_frame["spatial_completeness_pct"]
        .ge(VALID_PIXEL_THRESHOLD_PCT)
        .mean()
        if event_frame["spatial_completeness_pct"].notna().any()
        else np.nan
    )

    if input_type.startswith("Gap-filled"):
        quality_flag = "Gap-filled diagnostic"

    elif (
        temporal_observation_pct >= 60
        and pd.notna(median_spatial_completeness)
        and median_spatial_completeness >= VALID_PIXEL_THRESHOLD_PCT
    ):
        quality_flag = "Reliability-qualified and interpretable"

    else:
        quality_flag = "Reliability-qualified but observation-limited"

    return {
        "Series": series_label,
        "Input": input_type,
        "First negative anomaly": first_detection,
        "Detection delay (days)": (
            (first_detection - EVENT_DATE).days
            if pd.notna(first_detection)
            else np.nan
        ),
        "Peak negative anomaly date": impact_date,
        "Peak residual (nW cm⁻² sr⁻¹)": impact_row["residual"],
        "Peak departure from forecast (%)": (
            100
            * impact_row["residual"]
            / impact_row["predicted_ensemble"]
            if impact_row["predicted_ensemble"] != 0
            else np.nan
        ),
        "Negative anomaly days (0–180)": len(negative_anomalies),
        "Mean model confidence on negative anomalies (%)": (
            negative_anomalies["decision_confidence_pct"].mean()
        ),
        "T50 date": t50_date,
        "T50 (days from impact)": (
            (t50_date - impact_date).days
            if pd.notna(t50_date)
            else np.nan
        ),
        "T80 date": t80_date,
        "T80 (days from impact)": (
            (t80_date - impact_date).days
            if pd.notna(t80_date)
            else np.nan
        ),
        "Observed days in Haiyan window (%)": temporal_observation_pct,
        "Median spatial completeness (%)": median_spatial_completeness,
        "Days at or above valid-pixel threshold (%)": (
            days_at_valid_pixel_threshold
        ),
        "Valid-pixel threshold (%)": VALID_PIXEL_THRESHOLD_PCT,
        "Quality flag": quality_flag,
    }


## 4. Gap-filled Black Marble replication

This first model follows the published input logic most closely: a 30-day rolling mean of gap-filled VNP46A2 is used to fit the three adaptive forecasters. It tests whether the expected-versus-observed design identifies a negative Haiyan anomaly. Because the input is reconstructed, temporal continuity must not be interpreted as direct observability.


In [14]:
# ============================================================
# 6.1 FIT GAP-FILLED FCNN, CNN, LSTM, AND ENSEMBLE
# ============================================================

(
    gap_models,
    gap_result,
    gap_training_report,
) = fit_adaptive_ensemble(
    prepared_frame=gap_filled_daily,
    series_label="Region VIII gap-filled",
)

gap_haiyan_summary = pd.DataFrame(
    [
        summarize_haiyan(
            result=gap_result,
            series_label="Region VIII gap-filled",
            input_type="Gap-filled Black Marble",
        )
    ]
)

display(
    gap_training_report.style.format(
        {
            "Final training MAE": "{:.4f}",
            "Final validation MAE": "{:.4f}",
        }
    )
)

display(gap_haiyan_summary)


Region VIII gap-filled: training FCNN for 70 epochs
Region VIII gap-filled: training CNN for 90 epochs
Region VIII gap-filled: training LSTM for 25 epochs


,Series,Model,Training windows,Validation windows,Final training MAE,Final validation MAE
0,Region VIII gap-filled,FCNN,141,36,0.0596,0.0456
1,Region VIII gap-filled,CNN,141,36,0.0592,0.1337
2,Region VIII gap-filled,LSTM,141,36,0.1375,0.1349


,Series,Input,First negative anomaly,Detection delay (days),Peak negative anomaly date,Peak residual (nW cm⁻² sr⁻¹),Peak departure from forecast (%),Negative anomaly days (0–180),Mean model confidence on negative anomalies (%),T50 date,T50 (days from impact),T80 date,T80 (days from impact),Observed days in Haiyan window (%),Median spatial completeness (%),Days at or above valid-pixel threshold (%),Valid-pixel threshold (%),Quality flag
0,Region VIII gap-filled,Gap-filled Black Marble,2014-01-24,77,2014-02-03,-0.062632,-31.40869,14,66.666667,2014-02-27,24,2014-03-25,50,100.0,NaN,NaN,50.0,Gap-filled diagnostic


In [15]:
# ============================================================
# 6.2 SHARED FORECAST AND ANOMALY PLOT
# ============================================================

def plot_forecast_anomaly(
    result,
    title,
    support_rows,
):
    figure = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.20, 0.80],
        subplot_titles=(
            "Observation support",
            "Observed and expected nighttime lights",
        ),
    )

    figure.add_trace(
        go.Heatmap(
            x=result.index,
            y=[item["label"] for item in support_rows],
            z=np.vstack(
                [
                    result[item["column"]].to_numpy()
                    for item in support_rows
                ]
            ),
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>"
                "%{y}: %{z:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    figure.add_trace(
        go.Scatter(
            x=result.index,
            y=result["observed"],
            mode="lines",
            name="Observed 30-day NTL",
            line=dict(
                color=OBSERVED_COLOR,
                width=2.4,
            ),
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

    figure.add_trace(
        go.Scatter(
            x=result.index,
            y=result["predicted_ensemble"],
            mode="lines",
            name="Expected NTL (ensemble)",
            line=dict(
                color=PREDICTED_COLOR,
                width=2.2,
                dash="dash",
            ),
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

    anomaly_points = result[
        result["ensemble_anomaly"].eq(True)
    ]

    figure.add_trace(
        go.Scatter(
            x=anomaly_points.index,
            y=anomaly_points["observed"],
            mode="markers",
            name="Top-quartile prediction error",
            marker=dict(
                color=ANOMALY_COLOR,
                size=7,
                symbol="circle-open",
                line=dict(width=1.5),
            ),
            customdata=np.column_stack(
                [
                    anomaly_points["residual"],
                    anomaly_points["decision_confidence_pct"],
                ]
            ),
            hovertemplate=(
                "%{x|%d %b %Y}<br>"
                "Observed: %{y:.3f}<br>"
                "Residual: %{customdata[0]:.3f}<br>"
                "Model agreement: %{customdata[1]:.0f}%"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

    for row_number in (1, 2):
        figure.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line=dict(
                color=EVENT_LINE_COLOR,
                width=2,
                dash="dash",
            ),
            row=row_number,
            col=1,
        )

    figure.add_annotation(
        x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
        y=0.05,
        xref="x2",
        yref="y2 domain",
        text="Haiyan",
        showarrow=False,
        xanchor="left",
        font=dict(
            color=EVENT_LINE_COLOR,
            size=16,
        ),
    )

    figure.update_xaxes(
        range=[DISPLAY_START, DISPLAY_END],
        title_text="Date",
        row=2,
        col=1,
    )

    figure.update_yaxes(
        title_text="Mean radiance<br>(nW cm⁻² sr⁻¹)",
        row=2,
        col=1,
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1200,
        height=650,
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=24),
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.04,
            xanchor="center",
            x=0.5,
            font=dict(size=15),
        ),
        coloraxis=dict(
            colorscale=SC_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title="Support (%)",
                thickness=18,
            ),
        ),
        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),
        margin=dict(l=105, r=100, t=130, b=70),
        hovermode="x unified",
    )

    return figure


In [16]:
# ------------------------------------------------------------
# GAP-FILLED MODEL OUTPUT AND HAIYAN ANOMALIES
# ------------------------------------------------------------

fig_gap_forecast = plot_forecast_anomaly(
    result=gap_result,
    title=(
        "Chakraborty–Stokes replication: "
        "gap-filled Region VIII nighttime lights"
    ),
    support_rows=[
        {
            "label": "Gap-filled availability",
            "column": "temporal_support_pct",
        },
    ],
)

fig_gap_forecast.show()


**Interpretation.** A negative residual means observed 30-day NTL is below the learned expected trajectory. The 30-day smoother necessarily attenuates abrupt daily changes and can delay the apparent anomaly. Red markers show relative top-quartile errors, not independently verified outage days. The dense gap-filled record supports model continuity but does not establish that the satellite directly observed the ground around landfall.


## 5. Reliability-qualified GHSL application

The same architectures, training cutoff, epochs, forecast aggregation, ensemble weights, and anomaly threshold are now applied to all eight GHSL thematic masks. Only the input observation rule changes. Each mask is modelled independently so that its expected NTL level and temporal variation are learned from its own pre-Haiyan record.

This step tests method transfer, not algorithm improvement. A model can return a forecast whenever the preceding 60-day reliability-qualified rolling series is complete, but a recovery claim remains conditional on observed-day support and valid-pixel coverage.


In [17]:
# ============================================================
# 7.1 FIT THE SAME ENSEMBLE TO ALL GHSL THEMATIC MASKS
# ============================================================

rq_models = {}
rq_results = {}
rq_training_reports = []
rq_haiyan_rows = []

for mask_label, prepared_frame in rq_daily.items():
    (
        fitted_models,
        fitted_result,
        training_report,
    ) = fit_adaptive_ensemble(
        prepared_frame=prepared_frame,
        series_label=mask_label,
    )

    rq_models[mask_label] = fitted_models
    rq_results[mask_label] = fitted_result
    rq_training_reports.append(training_report)

    rq_haiyan_rows.append(
        summarize_haiyan(
            result=fitted_result,
            series_label=mask_label,
            input_type="Reliability-qualified DNB-BRDF",
        )
    )

rq_training_report = pd.concat(
    rq_training_reports,
    ignore_index=True,
)

rq_haiyan_summary = pd.DataFrame(rq_haiyan_rows)

display(
    rq_training_report.style.format(
        {
            "Final training MAE": "{:.4f}",
            "Final validation MAE": "{:.4f}",
        }
    )
)


G1 (codes 10–30): training FCNN for 70 epochs
G1 (codes 10–30): training CNN for 90 epochs
G1 (codes 10–30): training LSTM for 25 epochs
G2 (codes 11–30): training FCNN for 70 epochs
G2 (codes 11–30): training CNN for 90 epochs
G2 (codes 11–30): training LSTM for 25 epochs
G3 (codes 12–30): training FCNN for 70 epochs
G3 (codes 12–30): training CNN for 90 epochs
G3 (codes 12–30): training LSTM for 25 epochs
G4 (codes 13–30): training FCNN for 70 epochs
G4 (codes 13–30): training CNN for 90 epochs
G4 (codes 13–30): training LSTM for 25 epochs
G5 (codes 21–30): training FCNN for 70 epochs
G5 (codes 21–30): training CNN for 90 epochs
G5 (codes 21–30): training LSTM for 25 epochs
G6 (codes 22–30): training FCNN for 70 epochs
G6 (codes 22–30): training CNN for 90 epochs
G6 (codes 22–30): training LSTM for 25 epochs
G7 (codes 23–30): training FCNN for 70 epochs
G7 (codes 23–30): training CNN for 90 epochs
G7 (codes 23–30): training LSTM for 25 epochs
G8 (code 30): training FCNN for 70 epochs

,Series,Model,Training windows,Validation windows,Final training MAE,Final validation MAE
0,G1 (codes 10–30),FCNN,150,38,0.0613,0.0401
1,G1 (codes 10–30),CNN,150,38,0.0697,0.1862
2,G1 (codes 10–30),LSTM,150,38,0.1362,0.1310
3,G2 (codes 11–30),FCNN,150,38,0.0766,0.0602
4,G2 (codes 11–30),CNN,150,38,0.0694,0.2230
5,G2 (codes 11–30),LSTM,150,38,0.0946,0.0799
6,G3 (codes 12–30),FCNN,150,38,0.0961,0.0383
7,G3 (codes 12–30),CNN,150,38,0.0651,0.1547
8,G3 (codes 12–30),LSTM,150,38,0.1506,0.1350
9,G4 (codes 13–30),FCNN,150,38,0.1282,0.0597


In [18]:
# ------------------------------------------------------------
# RELIABILITY-QUALIFIED G7 MODEL OUTPUT
# ------------------------------------------------------------

fig_g7_forecast = plot_forecast_anomaly(
    result=rq_results[g7_label],
    title=(
        "Reliability-qualified adaptive forecast: "
        f"{g7_label}"
    ),
    support_rows=[
        {
            "label": "Observed-day support",
            "column": "temporal_support_pct",
        },
        {
            "label": "Spatial completeness",
            "column": "spatial_completeness_pct",
        },
    ],
)

fig_g7_forecast.show()


**Interpretation.** The G7 panel is the reliability-qualified counterpart to the gap-filled benchmark. Forecast gaps identify times when a complete 60-day reliability-qualified input was unavailable. Observed NTL, expected NTL, residuals, and recovery metrics should be interpreted only after inspecting both observed-day support and spatial completeness.


## 6. Cross-comparison of Haiyan anomalies

The comparison table places observability before anomaly and recovery metrics. The gap-filled row is retained for algorithmic comparison but is always labelled diagnostic. A reliability-qualified row is labelled observation-limited when its median post-Haiyan spatial completeness does not reach the declared 50% valid-pixel threshold, even if the neural network returns a numerical anomaly.

T50 and T80 are derived from the negative ensemble residual after its post-Haiyan minimum. They identify the first date on which at least 50% or 80% of peak residual severity has closed for 14 consecutive retained days. These are model-residual recovery indicators, not percentages of electricity service restored.


In [19]:
# ============================================================
# 8.1 COMPARISON TABLE
# ============================================================

haiyan_comparison = pd.concat(
    [
        gap_haiyan_summary,
        rq_haiyan_summary,
    ],
    ignore_index=True,
)

comparison_columns = [
    "Series",
    "Input",
    "Quality flag",
    "Observed days in Haiyan window (%)",
    "Median spatial completeness (%)",
    "Days at or above valid-pixel threshold (%)",
    "First negative anomaly",
    "Detection delay (days)",
    "Peak negative anomaly date",
    "Peak residual (nW cm⁻² sr⁻¹)",
    "Peak departure from forecast (%)",
    "Negative anomaly days (0–180)",
    "Mean model confidence on negative anomalies (%)",
    "T50 (days from impact)",
    "T80 (days from impact)",
    "Valid-pixel threshold (%)",
]

display(
    haiyan_comparison[comparison_columns]
    .style.format(
        {
            "Observed days in Haiyan window (%)": "{:.1f}",
            "Median spatial completeness (%)": "{:.1f}",
            "Days at or above valid-pixel threshold (%)": "{:.1f}",
            "Peak residual (nW cm⁻² sr⁻¹)": "{:.3f}",
            "Peak departure from forecast (%)": "{:.1f}",
            "Mean model confidence on negative anomalies (%)": "{:.1f}",
            "Valid-pixel threshold (%)": "{:.0f}",
        },
        na_rep="—",
    )
)


,Series,Input,Quality flag,Observed days in Haiyan window (%),Median spatial completeness (%),Days at or above valid-pixel threshold (%),First negative anomaly,Detection delay (days),Peak negative anomaly date,Peak residual (nW cm⁻² sr⁻¹),Peak departure from forecast (%),Negative anomaly days (0–180),Mean model confidence on negative anomalies (%),T50 (days from impact),T80 (days from impact),Valid-pixel threshold (%)
0,Region VIII gap-filled,Gap-filled Black Marble,Gap-filled diagnostic,100.0,—,—,2014-01-24 00:00:00,77.000000,2014-02-03 00:00:00,-0.063,-31.4,14,66.7,24.000000,50.000000,50
1,G1 (codes 10–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,88.4,36.5,45.9,—,—,2014-02-15 00:00:00,-0.063,-24.5,0,—,51.000000,51.000000,50
2,G2 (codes 11–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,88.4,36.6,45.9,—,—,2013-11-30 00:00:00,-0.038,-16.9,0,—,25.000000,31.000000,50
3,G3 (codes 12–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,87.3,37.2,44.2,—,—,2013-12-16 00:00:00,-0.112,-36.3,0,—,45.000000,114.000000,50
4,G4 (codes 13–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,87.3,37.5,43.1,—,—,2013-12-19 00:00:00,-0.135,-37.1,0,—,100.000000,111.000000,50
5,G5 (codes 21–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,86.2,37.3,43.1,—,—,2014-01-07 00:00:00,-0.048,-14.5,0,—,92.000000,92.000000,50
6,G6 (codes 22–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,85.6,33.6,40.9,—,—,2013-12-23 00:00:00,0.010,2.1,0,—,—,—,50
7,G7 (codes 23–30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,85.6,31.9,39.2,—,—,2013-12-23 00:00:00,0.158,28.2,0,—,—,—,50
8,G8 (code 30),Reliability-qualified DNB-BRDF,Reliability-qualified but observation-limited,75.1,32.1,38.1,—,—,2014-01-11 00:00:00,0.482,49.0,0,—,—,—,50


In [20]:
# ============================================================
# 8.2 NORMALIZED RESIDUAL COMPARISON
# ============================================================

fig_residuals = go.Figure()

gap_residual_pct = (
    100
    * gap_result["residual"]
    / gap_result["predicted_ensemble"]
)

fig_residuals.add_trace(
    go.Scatter(
        x=gap_residual_pct.index,
        y=gap_residual_pct,
        mode="lines",
        name="Gap-filled Region VIII",
        line=dict(
            color=GAP_FILLED_COLOR,
            width=2.5,
            dash="dash",
        ),
        connectgaps=False,
    )
)

mask_colors = [
    "#9E9E9E",
    "#7E57C2",
    "#5C6BC0",
    "#29B6F6",
    "#26A69A",
    "#66BB6A",
    RQ_COLOR,
    "#00695C",
]

for (
    mask_label,
    result,
), color in zip(rq_results.items(), mask_colors):
    residual_pct = (
        100
        * result["residual"]
        / result["predicted_ensemble"]
    )

    fig_residuals.add_trace(
        go.Scatter(
            x=residual_pct.index,
            y=residual_pct,
            mode="lines",
            name=mask_label,
            line=dict(
                color=color,
                width=(3.0 if mask_label == g7_label else 1.5),
            ),
            opacity=(1.0 if mask_label == g7_label else 0.65),
            connectgaps=False,
        )
    )

fig_residuals.add_hline(
    y=0,
    line=dict(
        color=LOW_SUPPORT_COLOR,
        width=1.5,
        dash="dot",
    ),
)

fig_residuals.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color=EVENT_LINE_COLOR,
        width=2,
        dash="dash",
    ),
)

fig_residuals.add_annotation(
    x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
    y=0.05,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        color=EVENT_LINE_COLOR,
        size=16,
    ),
)

fig_residuals.update_xaxes(
    range=[
        EVENT_DATE - pd.Timedelta(days=180),
        HAIYAN_WINDOW_END,
    ],
    title_text="Date",
)

fig_residuals.update_yaxes(
    title_text="Observed departure from expected NTL (%)",
)

fig_residuals.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=620,
    title=dict(
        text="Haiyan anomaly relative to each adaptive model forecast",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=14),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=105, r=80, t=150, b=70),
    hovermode="x unified",
)

fig_residuals.show()


In [21]:
# ============================================================
# 8.3 NOTEBOOK-READY INTERPRETATION
# ============================================================

gap_row = haiyan_comparison.loc[
    haiyan_comparison["Series"].eq("Region VIII gap-filled")
].iloc[0]

g7_row = haiyan_comparison.loc[
    haiyan_comparison["Series"].eq(g7_label)
].iloc[0]

gap_delay = gap_row["Detection delay (days)"]
g7_delay = g7_row["Detection delay (days)"]

interpretation = f'''
### 6.1 Preliminary interpretation

The gap-filled benchmark first identifies a negative anomaly **{gap_delay:.0f} days** after Haiyan, while the reliability-qualified G7 model identifies one **{g7_delay:.0f} days** after landfall. These delays include the attenuation and lag imposed by the published 30-day rolling average.

For G7, directly observed NTL is available on **{g7_row['Observed days in Haiyan window (%)']:.1f}%** of days from landfall through day 180, but median spatial completeness is **{g7_row['Median spatial completeness (%)']:.1f}%**. Its result is therefore classified as **{g7_row['Quality flag'].lower()}** under the declared {VALID_PIXEL_THRESHOLD_PCT:.0f}% valid-pixel rule.

The numerical contrast indicates how the adaptive detector responds to reconstructed continuity versus reliability-qualified inputs. It does not by itself establish which trajectory represents electricity restoration. The reliability-qualified result can support a recovery claim only where observability is adequate; otherwise the correct outcome is “not observable,” not “not recovered.”
'''

display(Markdown(interpretation))



### 6.1 Preliminary interpretation

The gap-filled benchmark first identifies a negative anomaly **77 days** after Haiyan, while the reliability-qualified G7 model identifies one **nan days** after landfall. These delays include the attenuation and lag imposed by the published 30-day rolling average.

For G7, directly observed NTL is available on **85.6%** of days from landfall through day 180, but median spatial completeness is **31.9%**. Its result is therefore classified as **reliability-qualified but observation-limited** under the declared 50% valid-pixel rule.

The numerical contrast indicates how the adaptive detector responds to reconstructed continuity versus reliability-qualified inputs. It does not by itself establish which trajectory represents electricity restoration. The reliability-qualified result can support a recovery claim only where observability is adequate; otherwise the correct outcome is “not observable,” not “not recovered.”


## 7. Conclusions and limitations

- The notebook reproduces the published 30-day smoothing, 60-to-30-day multi-step forecasting, three-model neural-network ensemble, median overlap aggregation, residual direction and severity, top-quartile anomaly threshold, and model-agreement indicator.
- The pre-Haiyan training record is necessarily shorter than the study’s preferred three-year minimum. Validation error and forecast stability must therefore be reported with every anomaly result.
- Gap-filled NTL provides the closest replication of the paper’s dense input but cannot establish daily observability. It remains a diagnostic comparison.
- Reliability-qualified NTL uses only the supplied directly observed GHSL-masked summaries. The 30-day calculation requires at least 18 observed days and does not interpolate missing daily values.
- The regional gap-filled series and GHSL-masked Samar–Leyte series do not have identical spatial support. Differences combine input treatment and spatial domain and should not be attributed to gap filling alone.
- Pixel-level percentile clipping cannot be recreated from daily summary tables. The analysis preserves any upstream processing and does not infer a clipped mean from `NTL_p95`.
- The 30-day rolling average can attenuate and delay an abrupt disaster signal. Detection delay is therefore partly methodological rather than purely physical.
- A numerical forecast or recovery metric is not automatically interpretable. Observed-day support, the 50% valid-pixel threshold, GHSL mask, training baseline, event window, and missingness are carried into the comparison table.

**Observability first, interpretation second, recovery metrics third. “Not recovered” and “not observable” remain distinct outcomes.**
